# PV6.6：PV6.5 配方 + val648 玩家 + seed 23

- 第一格已設好 `ARM = 'pv66'`、`MAX_STEPS = 0`，直接正式訓練。
- 第三格把 val648 串接進訓練資料（只在 `STAGE = 'train'` 時執行），重開 runtime 後照常全部執行即可。
- 練完改 `STAGE = 'infer'`，下載 `runs/pv66/` 的 `history.csv` 與 `emb-cache/`。

# 序列模型訓練與推論

1. 把 `seq_bundle.zip` 上傳到這個帳號的 `MyDrive/seqt4/`。
2. 在 Colab 左側「Secrets」加入唯讀的 `GH_TOKEN`，並開啟 notebook 存取權。
3. 執行階段選 L4 或更高的 GPU（T4 跑不動），改第一格的 `ARM`、`COMMIT`、`POOL`、`STAGE`，再執行全部 cell。
4. 輸出寫在 `MyDrive/seqt4/runs/<ARM>/`。中斷或換帳號時，把 `runs/` 複製到新帳號的同一路徑，重跑即可接續。
5. 推論完成後下載 `runs/<ARM>/` 的 `history.csv` 與 `emb-cache/`。

In [ ]:
ARM = 'pv66'  # PV6.6：PV6.5 配方 + val648 玩家 + seed 23
COMMIT = 'main'  # 填 push 後的 commit hash，接續訓練時保持相同
POOL = 'all'  # 'target' 或 'all'，依 newdev 結果選擇
STAGE = 'infer'  # 'train' 或 'infer'
EPOCHS = 12
MAX_STEPS = 0  # 正式訓練（不冒煙）
ARMS = {"pv6": ["--state-feats", "--patch-dir", "data/patch"], "pv63": ["--state-feats", "--patch-dir", "data/patch", "--conv-kernel", "9", "--spatial-bias"], "pv64": ["--state-feats", "--patch-dir", "data/patch", "--conv-kernel", "9", "--spatial-bias", "--prefix-crop-prob", "0.2", "--prefix-crop-range", "80", "150", "--color-balance"], "pv65": ["--state-feats", "--patch-dir", "data/patch", "--conv-kernel", "9", "--spatial-bias", "--augment-d4", "--val-tta", "8"], "state": ["--state-feats"], "stateconf": ["--state-feats", "--conv-kernel", "9"]}
ARMS['pv66'] = ARMS['pv65'] + ['--seed', '23']
print(ARM, ARMS[ARM])

pv66 ['--state-feats', '--patch-dir', 'data/patch', '--conv-kernel', '9', '--spatial-bias', '--augment-d4', '--val-tta', '8', '--seed', '23']


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, subprocess, sys
B = '/content/drive/MyDrive/seqt4'
RUN = os.path.join(B, 'runs', ARM + ('-smoke' if MAX_STEPS else ''))
os.makedirs(RUN, exist_ok=True)
REPO, DATA = '/content/aicup-2026-go', '/content/seq'


def git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode:
        print(r.stderr.replace(userdata.get('GH_TOKEN'), '***'))
        raise SystemExit('git %s 失敗' % args[0])
    return r.stdout.strip()


if not os.path.isdir(os.path.join(DATA, 'data')):
    subprocess.run(['unzip', '-oq', B + '/seq_bundle.zip', 'data/*', 'tests/*', '-d', DATA], check=True)
if not os.path.exists(REPO):
    git('clone', '-q', 'https://%s@github.com/Jx-study/aicup-2026-go.git' % userdata.get('GH_TOKEN'), REPO)
os.chdir(REPO)
git('fetch', '-q', 'origin')
git('checkout', '-q', COMMIT if COMMIT != 'main' else 'origin/main')
for name in ('data', 'tests'):
    if os.path.islink(name):
        os.unlink(name)
    os.symlink(os.path.join(DATA, name), name)
print(git('log', '-1', '--oneline'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sgfmill'], check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
d6bd18e feat(train_seq_emb): apply D4 transforms to local patches in training and TTA
name, memory.total [MiB]
NVIDIA L4, 23034 MiB


In [ ]:
# 把 val648 串接進訓練資料（train 與 patch 依相同順序），原檔保留為 *_orig.npz
import numpy as np, shutil
if STAGE == 'train':
    n_total = {}
    for d, tr, va in (('data/seq', 'train.npz', 'val648.npz'), ('data/patch', 'train_patch.npz', 'val648_patch.npz')):
        src, bak = os.path.join(d, tr), os.path.join(d, tr.replace('.npz', '_orig.npz'))
        if not os.path.exists(bak):
            shutil.move(src, bak)
        with np.load(bak) as a, np.load(os.path.join(d, va)) as b:
            out = {k: (np.concatenate([a[k], b[k]]) if a[k].ndim else a[k]) for k in a.files}
        np.savez(src, **out)
        n_total[d] = len(out['key'])
        print(d, {k: v.shape for k, v in out.items()}, flush=True)
    assert n_total['data/seq'] == n_total['data/patch'] == 918278, n_total
    print('合併完成：918,278 局')

In [ ]:
def run(cmd, log):
    print(' '.join(cmd), flush=True)
    with open(log, 'a') as f:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='', flush=True)
            f.write(line)
    if p.wait():
        raise SystemExit('指令失敗，exit code %d' % p.returncode)

extra = ARMS[ARM] + (['--pool-mode', 'all'] if POOL == 'all' else [])
ckpt = os.path.join(RUN, 'best_model.pth')
if STAGE == 'train':
    head, commit_file = git('rev-parse', '--short', 'HEAD'), os.path.join(RUN, 'COMMIT')
    if os.path.exists(commit_file) and open(commit_file).read().strip() != head:
        raise SystemExit('RUN 已用 %s 訓練，這次是 %s；COMMIT 要與接續前相同' % (open(commit_file).read().strip(), head))
    with open(commit_file, 'w') as f:
        f.write(head + '\n')
    run([sys.executable, '-u', '-m', 'aicup.train_seq_emb', '--seq-dir', 'data/seq',
         '--ckpt-dir', RUN, '--epochs', str(1 if MAX_STEPS else EPOCHS), *extra,
         *(['--max-steps', str(MAX_STEPS), '--log-every', '100'] if MAX_STEPS else [])], RUN + '/train.log')
else:
    tta = ['--tta', '8'] if '--augment-d4' in extra else []
    pd_args = (['--patch-dir', 'data/patch'] if '--patch-dir' in extra else []) + tta
    for split in ('newdev', 'val648'):
        run([sys.executable, '-u', '-m', 'aicup.infer_seq_emb', '--mode', 'packed', '--ckpt', ckpt,
             '--seq-dir', 'data/seq', '--split', split, *pd_args,
             '--out', os.path.join(RUN, 'emb-cache', split + '_emb.npz')], RUN + '/infer.log')
    run([sys.executable, '-u', '-m', 'aicup.infer_seq_emb', '--mode', 'packed', '--ckpt', ckpt,
         '--seq-npz', 'data/seq/hold.npz', *pd_args,
         '--out', os.path.join(RUN, 'emb-cache', 'hold_emb.npz')], RUN + '/infer.log')
    run([sys.executable, '-u', '-m', 'aicup.infer_seq_emb', '--mode', 'test', '--ckpt', ckpt,
         '--groups', 'all', '--tests-dir', 'tests', '--out-dir', os.path.join(RUN, 'emb-cache'), *tta],
        RUN + '/infer.log')

/usr/bin/python3 -u -m aicup.infer_seq_emb --mode packed --ckpt /content/drive/MyDrive/seqt4/runs/pv66/best_model.pth --seq-dir data/seq --split newdev --patch-dir data/patch --tta 8 --out /content/drive/MyDrive/seqt4/runs/pv66/emb-cache/newdev_emb.npz
載入 /content/drive/MyDrive/seqt4/runs/pv66/best_model.pth（device=cuda, amp=True, cfg={'d_model': 256, 'nhead': 8, 'num_layers': 6, 'dim_feedforward': 1024, 'embed_dim': 256, 'max_len': 320, 'dropout': 0.1, 'head_dropout': 0.2, 'pool_mode': 'all', 'augment_d4': True, 'raw_color': False, 'state_feats': True, 'conv_kernel': 9, 'patch_size': 7, 'spatial_bias': True, 'stage_pool': False}）
packed 序列推論：data/seq/newdev.npz，54376 局，tta=8
寫出 /content/drive/MyDrive/seqt4/runs/pv66/emb-cache/newdev_emb.npz（54376 局，320.5 秒）
/usr/bin/python3 -u -m aicup.infer_seq_emb --mode packed --ckpt /content/drive/MyDrive/seqt4/runs/pv66/best_model.pth --seq-dir data/seq --split val648 --patch-dir data/patch --tta 8 --out /content/drive/MyDrive/seqt4/runs/pv66/emb

In [ ]:
from google.colab import runtime

drive.flush_and_unmount()
runtime.unassign()